In [1]:
!pip install openai python-dotenv

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.4 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openai]━━━━ 5/6 [openai]

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
pip install openai

  Using cached openai-3.6.0-py3-none-any.whl.metadata (41 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
Using cached openai-3.6.0-py3-none-any.whl (1.7 MB)
Using cached httpx2-2.12.0-py3-none-any.whl (95 kB)
Using cached httpcore2-2.12.0-py3-none-any.whl (83 kB)
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [openai]━━━━ 4/5 [openai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langsmith 0.8.8 requires websockets>=15.0, but you have websockets 13.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()


class LLMAdvisor:
    """
    Uses an LLM to analyze Bitcoin market conditions.

    IMPORTANT:
    The LLM does NOT execute trades.

    It only provides a structured recommendation.
    The TradingAgent and RiskManager remain responsible
    for deciding whether a trade can actually happen.
    """

    def __init__(self):

        self.api_key = os.getenv("LLM_API_KEY")

        if not self.api_key:
            raise ValueError(
                "LLM_API_KEY is missing from .env"
            )

        self.client = OpenAI(
            api_key=self.api_key
        )

        self.model = os.getenv(
            "LLM_MODEL",
            "gpt-5-mini"
        )

    def summarize_forecast(
        self,
        forecast_df,
        current_price
    ):
        """
        Convert the large 30-minute forecast dataset into
        a compact summary that can be safely provided to
        the LLM.

        Expected forecast columns:

            timestamp
            predicted_close
            lower_bound
            upper_bound

        Parameters
        ----------
        forecast_df : pandas.DataFrame
            Future BTC predictions.

        current_price : float
            Current BTC market price.

        Returns
        -------
        dict
            Forecast summary.
        """

        forecast = forecast_df.copy()

        forecast["timestamp"] = pd.to_datetime(
            forecast["timestamp"],
            utc=True
        )

        forecast = forecast.sort_values(
            "timestamp"
        )

        if forecast.empty:
            raise ValueError(
                "Forecast dataframe is empty."
            )

        # -----------------------------------------------------
        # Current time
        # -----------------------------------------------------

        now = forecast["timestamp"].min()

        # -----------------------------------------------------
        # Helper function
        # -----------------------------------------------------

        def get_forecast_at_hours(hours):
            """
            Find the forecast point closest to a
            specified number of hours in the future.
            """

            target_time = now + timedelta(
                hours=hours
            )

            differences = (
                forecast["timestamp"] - target_time
            ).abs()

            index = differences.idxmin()

            return forecast.loc[index]

        # -----------------------------------------------------
        # Get forecast horizons
        # -----------------------------------------------------

        horizons = {
            "24_hours": 24,
            "7_days": 7 * 24,
            "30_days": 30 * 24,
            "90_days": 90 * 24,
            "1_year": 365 * 24,
            "3_years": 3 * 365 * 24
        }

        summary = {}

        for name, hours in horizons.items():

            row = get_forecast_at_hours(hours)

            predicted_price = float(
                row["predicted_close"]
            )

            lower = float(
                row["lower_bound"]
            )

            upper = float(
                row["upper_bound"]
            )

            predicted_return = (
                (predicted_price - current_price)
                / current_price
                * 100
            )

            summary[name] = {
                "predicted_price": round(
                    predicted_price,
                    2
                ),

                "predicted_return_pct": round(
                    predicted_return,
                    2
                ),

                "lower_bound": round(
                    lower,
                    2
                ),

                "upper_bound": round(
                    upper,
                    2
                )
            }

        return summary
    # =========================================================
    # MARKET ANALYSIS
    # =========================================================

    def analyze_market(
        self,
        price,
        rsi,
        macd,
        macd_signal,
        atr,
        volume_ratio,
        sma_20=None,
        ema_20=None,
        recent_return_pct=None,
        forecast_summary=None
    ):
        """
        Analyze the current Bitcoin market.

        Returns a structured dictionary containing:

        - market_regime
        - recommendation
        - confidence
        - reason
        - suggested_atr_multiplier
        """

        market_data = {
            "price": price,
            "rsi": rsi,
            "macd": macd,
            "macd_signal": macd_signal,
            "atr": atr,
            "volume_ratio": volume_ratio,
            "sma_20": sma_20,
            "ema_20": ema_20,
            "recent_return_pct": recent_return_pct
        }

        prompt = f"""
You are a Bitcoin trading market-analysis assistant.

Analyze the following technical market data.

Market data:
{json.dumps(market_data, indent=2)}

LONG-TERM BTC FORECAST:

{json.dumps(
    forecast_summary,
    indent=2
)}


Determine:

1. Market regime:
   - bullish
   - bearish
   - sideways
   - high_volatility

2. Recommended strategy:
   - DCA
   - SWING
   - HOLD

3. Confidence from 0 to 1.

4. A short explanation.

5. A reasonable ATR stop-loss multiplier between
   1.0 and 3.0.

IMPORTANT:
You are only providing analysis.
You are NOT executing a trade.

Return ONLY valid JSON in exactly this format:

{{
    "market_regime": "bullish",
    "recommendation": "SWING",
    "confidence": 0.75,
    "reason": "Short explanation",
    "suggested_atr_multiplier": 1.5
}}
"""
        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        text = response.output_text.strip()

        try:
            result = json.loads(text)

        except json.JSONDecodeError as exc:

            raise ValueError(
                f"LLM returned invalid JSON: {text}"
            ) from exc

        return self._validate_result(result)

    # =========================================================
    # VALIDATE RESPONSE
    # =========================================================

    def _validate_result(self, result):
        """
        Validate and sanitize the LLM response.
        """

        required_fields = [
            "market_regime",
            "recommendation",
            "confidence",
            "reason",
            "suggested_atr_multiplier"
        ]

        for field in required_fields:

            if field not in result:
                raise ValueError(
                    f"LLM response missing: {field}"
                )

        valid_regimes = {
            "bullish",
            "bearish",
            "sideways",
            "high_volatility"
        }

        valid_recommendations = {
            "DCA",
            "SWING",
            "HOLD"
        }

        regime = result["market_regime"].lower()

        recommendation = (
            result["recommendation"].upper()
        )

        if regime not in valid_regimes:
            raise ValueError(
                f"Invalid market regime: {regime}"
            )

        if recommendation not in valid_recommendations:
            raise ValueError(
                f"Invalid recommendation: "
                f"{recommendation}"
            )

        confidence = float(
            result["confidence"]
        )

        confidence = max(
            0.0,
            min(1.0, confidence)
        )

        atr_multiplier = float(
            result["suggested_atr_multiplier"]
        )

        # Never allow the LLM to suggest an
        # extreme stop-loss multiplier.
        atr_multiplier = max(
            1.0,
            min(3.0, atr_multiplier)
        )

        return {
            "market_regime": regime,
            "recommendation": recommendation,
            "confidence": confidence,
            "reason": str(result["reason"]),
            "suggested_atr_multiplier": atr_multiplier
        }

In [5]:
import os
import json

from datetime import timedelta

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()


class LLMAdvisor:
    """
    Uses an LLM to analyze Bitcoin market conditions.

    The LLM receives:
        - Current technical indicators
        - Recent market performance
        - Long-term BTC price forecast

    IMPORTANT:
    The LLM does NOT execute trades.

    It only provides a structured recommendation.

    The TradingAgent and RiskManager remain responsible
    for deciding whether a trade can actually happen.
    """

    def __init__(self):

        self.api_key = os.getenv("LLM_API_KEY")

        if not self.api_key:
            raise ValueError(
                "LLM_API_KEY is missing from .env"
            )

        self.client = OpenAI(
            api_key=self.api_key
        )

        self.model = os.getenv(
            "LLM_MODEL",
            "gpt-5-mini"
        )

    # =========================================================
    # FORECAST SUMMARY
    # =========================================================

    def summarize_forecast(
        self,
        forecast_df,
        current_price
    ):
        """
        Convert the large 30-minute forecast dataset into
        a compact summary that can be safely provided to
        the LLM.

        Expected forecast columns:

            timestamp
            predicted_close
            lower_bound
            upper_bound

        Parameters
        ----------
        forecast_df : pandas.DataFrame
            Future BTC predictions.

        current_price : float
            Current BTC market price.

        Returns
        -------
        dict
            Forecast summary.
        """

        forecast = forecast_df.copy()

        forecast["timestamp"] = pd.to_datetime(
            forecast["timestamp"],
            utc=True
        )

        forecast = forecast.sort_values(
            "timestamp"
        )

        if forecast.empty:
            raise ValueError(
                "Forecast dataframe is empty."
            )

        # -----------------------------------------------------
        # Current time
        # -----------------------------------------------------

        now = forecast["timestamp"].min()

        # -----------------------------------------------------
        # Helper function
        # -----------------------------------------------------

        def get_forecast_at_hours(hours):
            """
            Find the forecast point closest to a
            specified number of hours in the future.
            """

            target_time = now + timedelta(
                hours=hours
            )

            differences = (
                forecast["timestamp"] - target_time
            ).abs()

            index = differences.idxmin()

            return forecast.loc[index]

        # -----------------------------------------------------
        # Get forecast horizons
        # -----------------------------------------------------

        horizons = {
            "24_hours": 24,
            "7_days": 7 * 24,
            "30_days": 30 * 24,
            "90_days": 90 * 24,
            "1_year": 365 * 24,
            "3_years": 3 * 365 * 24
        }

        summary = {}

        for name, hours in horizons.items():

            row = get_forecast_at_hours(hours)

            predicted_price = float(
                row["predicted_close"]
            )

            lower = float(
                row["lower_bound"]
            )

            upper = float(
                row["upper_bound"]
            )

            predicted_return = (
                (predicted_price - current_price)
                / current_price
                * 100
            )

            summary[name] = {
                "predicted_price": round(
                    predicted_price,
                    2
                ),

                "predicted_return_pct": round(
                    predicted_return,
                    2
                ),

                "lower_bound": round(
                    lower,
                    2
                ),

                "upper_bound": round(
                    upper,
                    2
                )
            }

        return summary

    # =========================================================
    # MARKET ANALYSIS
    # =========================================================

    def analyze_market(
        self,
        price,
        rsi,
        macd,
        macd_signal,
        atr,
        volume_ratio,
        sma_20=None,
        ema_20=None,
        recent_return_pct=None,
        forecast_summary=None
    ):
        """
        Analyze the current Bitcoin market.

        Parameters
        ----------
        price : float
            Current BTC price.

        rsi : float
            RSI indicator.

        macd : float
            MACD value.

        macd_signal : float
            MACD signal.

        atr : float
            ATR value.

        volume_ratio : float
            Current volume relative to average volume.

        sma_20 : float, optional
            20-period SMA.

        ema_20 : float, optional
            20-period EMA.

        recent_return_pct : float, optional
            Recent BTC return percentage.

        forecast_summary : dict, optional
            Long-term BTC forecast summary.

        Returns
        -------
        dict
            Structured LLM recommendation.
        """

        # -----------------------------------------------------
        # Technical market data
        # -----------------------------------------------------

        market_data = {
            "price": price,
            "rsi": rsi,
            "macd": macd,
            "macd_signal": macd_signal,
            "atr": atr,
            "volume_ratio": volume_ratio,
            "sma_20": sma_20,
            "ema_20": ema_20,
            "recent_return_pct": recent_return_pct
        }

        # -----------------------------------------------------
        # Forecast data
        # -----------------------------------------------------

        if forecast_summary is None:

            forecast_summary = {
                "status": "No forecast available"
            }

        # -----------------------------------------------------
        # Build prompt
        # -----------------------------------------------------

        prompt = f"""
You are a Bitcoin trading market-analysis assistant.

Analyze the current Bitcoin market using both:

1. Short-term technical indicators
2. Long-term model forecast

CURRENT MARKET DATA:

{json.dumps(
    market_data,
    indent=2
)}


LONG-TERM BTC FORECAST:

{json.dumps(
    forecast_summary,
    indent=2
)}


Determine:

1. Market regime:
   - bullish
   - bearish
   - sideways
   - high_volatility

2. Recommended strategy:
   - DCA
   - SWING
   - HOLD

3. Confidence from 0 to 1.

4. A short explanation that considers BOTH:
   - current technical conditions
   - forecast direction

5. A reasonable ATR stop-loss multiplier between
   1.0 and 3.0.

IMPORTANT:

The forecast is a MODEL PREDICTION, not guaranteed
future market data.

Do not assume that a bullish long-term forecast
means an immediate BUY.

Short-term technical indicators should be considered
for short-term trading decisions.

The long-term forecast should provide additional
context rather than override the Risk Manager.

The LLM does NOT execute trades.

Return ONLY valid JSON in exactly this format:

{{
    "market_regime": "bullish",
    "recommendation": "SWING",
    "confidence": 0.75,
    "reason": "Short explanation",
    "suggested_atr_multiplier": 1.5
}}
"""

        # -----------------------------------------------------
        # Call OpenAI
        # -----------------------------------------------------

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        text = response.output_text.strip()

        # -----------------------------------------------------
        # Parse JSON
        # -----------------------------------------------------

        try:

            result = json.loads(text)

        except json.JSONDecodeError as exc:

            raise ValueError(
                f"LLM returned invalid JSON: {text}"
            ) from exc

        # -----------------------------------------------------
        # Validate result
        # -----------------------------------------------------

        return self._validate_result(result)

    # =========================================================
    # VALIDATE RESPONSE
    # =========================================================

    def _validate_result(self, result):
        """
        Validate and sanitize the LLM response.
        """

        required_fields = [
            "market_regime",
            "recommendation",
            "confidence",
            "reason",
            "suggested_atr_multiplier"
        ]

        for field in required_fields:

            if field not in result:

                raise ValueError(
                    f"LLM response missing: {field}"
                )

        valid_regimes = {
            "bullish",
            "bearish",
            "sideways",
            "high_volatility"
        }

        valid_recommendations = {
            "DCA",
            "SWING",
            "HOLD"
        }

        regime = (
            str(result["market_regime"])
            .lower()
        )

        recommendation = (
            str(result["recommendation"])
            .upper()
        )

        if regime not in valid_regimes:

            raise ValueError(
                f"Invalid market regime: {regime}"
            )

        if recommendation not in valid_recommendations:

            raise ValueError(
                f"Invalid recommendation: "
                f"{recommendation}"
            )

        confidence = float(
            result["confidence"]
        )

        confidence = max(
            0.0,
            min(1.0, confidence)
        )

        atr_multiplier = float(
            result["suggested_atr_multiplier"]
        )

        # Never allow the LLM to suggest an
        # extreme stop-loss multiplier.

        atr_multiplier = max(
            1.0,
            min(3.0, atr_multiplier)
        )

        return {
            "market_regime": regime,

            "recommendation": recommendation,

            "confidence": confidence,

            "reason": str(
                result["reason"]
            ),

            "suggested_atr_multiplier":
                atr_multiplier
        }

In [6]:
import os

print("API key exists:", bool(os.getenv("LLM_API_KEY")))

API key exists: True


In [7]:
from openai import OpenAI

api_key = os.getenv("LLM_API_KEY")
client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-5.6",
    input="Say hello in one sentence."
)

print(response.output_text)

Hello!


In [10]:
llm = LLMAdvisor()

current_price = 110000

forecast_df = pd.read_csv(
    "../data/btc_3year_forecast_30min.csv"
)

forecast_summary = llm.summarize_forecast(
    forecast_df=forecast_df,
    current_price=current_price
)

decision = llm.analyze_market(
    price=110000,
    rsi=62,
    macd=450,
    macd_signal=400,
    atr=1500,
    volume_ratio=1.8,
    sma_20=107000,
    ema_20=108000,
    recent_return_pct=3.2,
    forecast_summary = forecast_summary
)

print(decision)

{'market_regime': 'high_volatility', 'recommendation': 'DCA', 'confidence': 0.55, 'reason': 'Short-term technicals are bullish: price > SMA20/EMA20, RSI 62, MACD > signal and elevated volume with a recent +3.2% move, indicating momentum. However the model forecasts a sharp 24h drop (~-29%) and shows extremely wide, inconsistent longer-horizon bounds, creating significant downside risk or model instability. Given the bullish short-term setup but high model-driven tail risk and uncertainty, prefer risk-managed DCA while monitoring for a volatility-triggered reversal.', 'suggested_atr_multiplier': 2.0}
